# 35. An unrelated probe task: EuroSAT

Notebook 29 found that the generator learns an unseen task 6.62 points **better** after
eight tasks than it does fresh. That reading has a hole in it, and `uncle/probe.py` names
the hole itself: every probe task so far comes from the same dataset as the stream. So the
gain may be transfer from similar Tiny ImageNet classes rather than plasticity.

This notebook closes that hole. EuroSAT is satellite imagery, unrelated to Tiny ImageNet,
and it has exactly ten classes, which is the only way a probe task can fit the generator's
fixed output layout.

**Second pass.** The first run, on 28 October, left the question open: both probes rose, and
neither rose past its own noise floor. Three things changed since.

| Change | Why |
| --- | --- |
| The learning rate is chosen on a validation split, and the result is read from a separate test split | The first run chose the rate on test accuracy, which picks the luckiest number and widens the spread. The plan of 22 October asks for this. |
| Eleven repetitions at both probe points, instead of five and two | The first run's floor was 14.66 for EuroSAT against a change of +7.76. Eleven at both points cuts that floor to about 7.4. |
| Only the full arm runs | The code-only arm read chance in four places out of four. It is settled. |

Splits followed the 22 October plan: 800 training, 200 validation and 2,000 test images
for EuroSAT, with equal numbers from every class.

**Third pass, and what this notebook is set to right now.** The second pass answered
nothing about plasticity but raised a new question. Its measured change on Tiny ImageNet
task 2 collapsed from +5.76 to +1.18. Scoring the same runs again under the old rate rule
gave +1.11, so the rate rule is not the cause. Two differences remain: the training split
fell from 5,000 images to 800, and the repetitions at t = 8 rose from two to eleven.

This pass changes one of them and nothing else. The training split goes back up. EuroSAT
is switched off, because it has no part in this question and costs half the session.

| Setting | Second pass | This pass |
| --- | --- | --- |
| Probe tasks | EuroSAT and Tiny ImageNet task 2 | Tiny ImageNet task 2 only |
| Training images per probe task | 800 | 4,800 |
| Validation images | 200 | 200 |
| Test images, task 2 | 500 | 500 |
| Rate chosen on | validation accuracy | validation accuracy, unchanged |
| Repetitions at t = 0 and t = 8 | 11 and 11 | 11 and 11, unchanged |
| Arms | full | full, unchanged |
| Seeds | 0 | 0, 1 and 2 |
| Runs | 132 | 198, of which 66 are already saved |

The training set is 4,800 rather than the 5,000 of the earlier runs, because 200 images are
held back for validation and those must not be trained on. That 4 percent is small beside
the difference being tested.

**What it decides**

| Tiny ImageNet task 2 at t = 8 | Reading |
| --- | --- |
| the change returns to about +5 | the training split is what moved it, and the earlier gains were real at 5,000 images |
| the change stays near +1 | the split is not the cause either, and what remains is the probe-task set: the 21 October gain would then rest on averaging three tasks |
| somewhere between | both contribute, and the next test would be the probe-task set |

Either answer is useful. This is the first run in the EuroSAT line that cannot come back
saying nothing.

**Seeds 1 and 2 were added after seed 0 came back.** Seed 0 read +2.04 against a floor of
2.63, which answered the split question and left the size of the change unresolved. Two
more seeds were chosen over more repetitions on seed 0, because the settings are fixed
before the data arrives rather than after. Nothing else changed, so seed 0's 66 saved runs
are skipped and only the 132 new ones cost time.

**Switching back to EuroSAT.** Set `INCLUDE_EUROSAT = True` and put `TRAIN_PER_CLASS` back
to 80 in section 2. Give `RUN_LABEL` a new name so the two protocols never share a saved
file.

Finished probe runs are saved to R2 as they complete, so stopping and restarting the
notebook continues where it stopped. Expect roughly 40 minutes for the settings below.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-probe")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.probe import noise_floor, probe_once, snapshot, unchanged

import torch
from tqdm.auto import tqdm

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)

## 2. Settings

The Tiny ImageNet reference probe gets the same treatment as EuroSAT: same training size,
same validation split, same repetition counts. So the two probes differ by their images and
nothing else. The cell prints the run count and an honest time estimate before anything
starts.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"
ORDER = "order_01"
# Seed 0 ran first and came back at +2.04 against a floor of 2.63, which
# settled the training-split question and left the change unresolved. Seeds 1
# and 2 were then added at these exact settings. Because nothing else changed,
# RUN_LABEL stays the same and seed 0's finished runs are skipped on resume.
SEEDS = [0, 1, 2]

# Names this protocol. Every change to the settings below that would make the
# numbers incomparable needs a new label, because finished runs are matched by
# identity and a shared file would silently mix two protocols.
RUN_LABEL = "pass3_train4800"

# Part-finished results live in R2, not on the Colab disk, because that disk
# disappears when the session ends. Delete this file to force a clean rerun.
RESUME_KEY = f"uncle/eurosat_probe/{STUDY_ID}/{ORDER}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 10               # upload after this many new runs

CLASSES_HINT = 10             # classes per task, confirmed against the plan in section 4
EUROSAT = "eurosat"

# Off for this pass. The question here is whether the training split explains
# the second pass's collapse, and EuroSAT has no part in it. Leaving it on
# would double the session and add nothing. Switching it back on also means
# putting TRAIN_PER_CLASS back to 80 and choosing a new RUN_LABEL.
INCLUDE_EUROSAT = False

# The Tiny ImageNet probe task. The record of 17 October shows repeats inside
# one session vary by about 3 points and repeats across sessions by 8.4, so a
# comparison against stored numbers from another session would mix the probe
# task with the session. Both probe points here run in one session.
REFERENCE_TASKS = ["2"]

# A model late in a stream may need a different rate than a fresh one, so the
# probe tries a grid and keeps the best. A single fixed rate would confuse a
# rate mismatch with plasticity loss.
RATES = [1e-4, 3e-4, 1e-3]
STEPS = 250                   # never varied between probe points
BATCH_SIZE = 64
FEATURE_IMAGES = 2000         # for the effective rank

# Equal at both points. The floor is 1.96 * s * sqrt(1/n_0 + 1/n_t), so with
# the same count at both points it falls as 1 over the square root of that
# count. The first run's 5 and 2 gave a floor of 14.66 on EuroSAT, wider than
# any change worth finding. Eleven and eleven bring it near 7.4 at the same
# spread.
REPETITIONS_AT_ZERO = 11
REPETITIONS_LATER = 11

# (name, generator frozen, code trains). These names must match what the probe
# writes into each result, because the resume check compares them.
# The code-only arm is dropped. It read chance at both probe points on both
# probe tasks in the first run, so repeating it buys nothing and would cost
# half the session.
ARMS = [
    ("full", False, True),        # generator and code both train
]

# Per class. 480 training and 20 validation gives 4,800 and 200 in total,
# which is the 5,000-image split of the 17, 21 and 28 October runs with the
# validation images held back. The second pass used 80 and 20, the sizes the
# 22 October plan asks for, and this pass exists to find out whether that
# change is what moved the result.
#
# TEST_PER_CLASS applies to EuroSAT only and is unused while INCLUDE_EUROSAT
# is False. The Tiny ImageNet task keeps the project's usual test split, 500
# images, which was the same in both earlier passes.
TRAIN_PER_CLASS = 480
VAL_PER_CLASS = 20
TEST_PER_CLASS = 200
SPLIT_SEED = 7                # fixes which images land in which split

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
EUROSAT_ROOT = Path("/content/data") if IN_COLAB else REPO / "data"

PROBE_TASKS = ([EUROSAT] if INCLUDE_EUROSAT else []) + REFERENCE_TASKS
if not PROBE_TASKS:
    raise ValueError("No probe tasks. Turn EuroSAT on or name a reference task.")
TOTAL_RUNS = (len(PROBE_TASKS) * len(RATES) * len(ARMS)
              * (REPETITIONS_AT_ZERO + REPETITIONS_LATER) * len(SEEDS))

# Recorded with every run. The noise floor is built from the t = 0 repetitions,
# so if those ran on one GPU and a later point on another, the difference
# between the machines could look like a result.
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", GPU_NAME)
print(f"Probe runs: {TOTAL_RUNS}  ({len(PROBE_TASKS)} tasks, {len(RATES)} rates, "
      f"{len(ARMS)} arms, {REPETITIONS_AT_ZERO} and {REPETITIONS_LATER} repetitions, "
      f"{len(SEEDS)} seed(s))")
print(f"At roughly 0.6 minutes each that is about {TOTAL_RUNS * 0.6:.0f} minutes "
      "for a clean start. Runs already saved are skipped, and the grid cell "
      "below reports how many are actually new.")
print("Rate chosen on validation accuracy, result read from the test split.")
print(f"Training images per probe task: {CLASSES_HINT * TRAIN_PER_CLASS}, "
      f"validation {CLASSES_HINT * VAL_PER_CLASS}. Saving under: {RUN_LABEL}")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Build the images and the two models

`t = 8` is the seed's finished eight-task model from R2. `t = 0` is a generator built with
the same settings and the same seed that has learned nothing.

EuroSAT downloads itself, about 90 MB. Its old DFKI address is dead; torchvision 0.27 and
later point at a working mirror, so an older torchvision will fail here.

Both probe tasks are cut into disjoint, class-balanced splits by the same helper. The
validation split exists so the learning rate is never chosen on the numbers the result is
read from.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = list(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]

overlap = set(REFERENCE_TASKS) & set(ORDER_TASKS)
if overlap:
    raise ValueError(f"These probe tasks are in the learning order: {sorted(overlap)}")

needed = tuple(ORDER_TASKS + REFERENCE_TASKS)
tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "tasks")

def source_state(seed):
    """That seed's finished eight-task model, read from R2."""
    job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{seed}")
    state = storage.load(job)
    if state is None or state["report"]["status"] != "complete":
        raise RuntimeError(f"Source model for seed {seed} is not finished in R2.")
    return state

# Check every seed before any training starts, so a missing source model stops
# the notebook now rather than half an hour into the run.
for seed in SEEDS:
    source_state(seed)
print("Source models present for seeds:", SEEDS)

def build(seed, restore_source):
    """A model at one probe point. restore_source False gives the t = 0 model."""
    config = Config(**{**plan["config"], "seed": seed, "device": DEVICE,
                       "requests": tuple(("learn", t) for t in ORDER_TASKS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    if restore_source:
        checkpoint.restore(source_state(seed), hypernet=hypernet, uncle=trainer)
    return target, hypernet, trainer

In [ ]:
from torchvision.datasets import EuroSAT
from torchvision.transforms import ToTensor
from torch.utils.data import Subset

def balanced_split(labels, counts, seed):
    """Row numbers for several disjoint splits, equal numbers from every class.

    `labels` holds one label per row and `counts` says how many images per
    class each split gets. Working on the labels alone means no image is read
    here, so this stays fast on a dataset of 27,000 files.
    """
    by_class = {}
    for row, label in enumerate(labels):
        by_class.setdefault(int(label), []).append(row)
    if len(by_class) != CLASSES:
        raise RuntimeError(f"Expected {CLASSES} classes, found {len(by_class)}.")
    stream = torch.Generator().manual_seed(seed)
    splits = [[] for _ in counts]
    for label, rows in sorted(by_class.items()):
        if len(rows) < sum(counts):
            raise RuntimeError(f"Class {label} has {len(rows)} images, needs {sum(counts)}.")
        order = [rows[j] for j in torch.randperm(len(rows), generator=stream).tolist()]
        at = 0
        for which, how_many in enumerate(counts):
            splits[which] += order[at:at + how_many]
            at += how_many
    taken = [row for split in splits for row in split]
    assert len(set(taken)) == len(taken), "An image landed in two splits."
    return splits

assert CLASSES_HINT == CLASSES, f"Settings assume {CLASSES_HINT} classes, the plan says {CLASSES}"
probe_sets = {}

if INCLUDE_EUROSAT:
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(), download=True)

    # The generator's output layout is fixed when it is built, and the backbone
    # expects 64x64. Both have to match or the probe measures the wrong thing.
    assert len(eurosat.classes) == CLASSES, f"EuroSAT has {len(eurosat.classes)} classes, need {CLASSES}"
    assert eurosat[0][0].shape == (3, 64, 64), f"Unexpected image shape {eurosat[0][0].shape}"

    # `samples` is a list of (path, label), so this reads no images.
    parts = balanced_split([label for _, label in eurosat.samples],
                           (TRAIN_PER_CLASS, VAL_PER_CLASS, TEST_PER_CLASS), SPLIT_SEED)
    probe_sets[EUROSAT] = dict(zip(("train", "val", "test"),
                                   (Subset(eurosat, part) for part in parts)))
else:
    print("EuroSAT is off for this pass.")

# The Tiny ImageNet task's training and validation splits are cut out of its
# own training images, so no image is ever both trained on and used to choose a
# learning rate. Its test split is the one the rest of the project uses, and it
# was the same 500 images in every pass, so test-set noise cannot explain any
# difference between passes.
for task in REFERENCE_TASKS:
    base = tasks[task]["train"]
    train_rows, val_rows = balanced_split(base.targets,
                                          (TRAIN_PER_CLASS, VAL_PER_CLASS), SPLIT_SEED)
    probe_sets[task] = {"train": Subset(base, train_rows),
                        "val": Subset(base, val_rows),
                        "test": tasks[task]["test"]}

for name, split in probe_sets.items():
    print(f"{name:<10} train {len(split['train']):>5}  val {len(split['val']):>5}  "
          f"test {len(split['test']):>5}")
print("Probe tasks:", PROBE_TASKS)

## 5. Run the grid

Every run snapshots the model, trains a fresh task code on a probe task, measures, then
puts the model back and checks it really is unchanged. The bar shows how many runs are
left and the time remaining. Stopping the cell loses at most the current run.

In [ ]:
import time

# A run is identified by everything that makes it different from another run.
def identity(run):
    return (run["seed_label"], run["point"], run["task"],
            run["learning_rate"], run["arm"], run["repetition"])

def probe_seed(task, repetition):
    """Fixes the batch order and the fresh code. Depends on the task and the
    repetition, never on the probe point, so two points differ only by the
    model's state."""
    return 12345 + 1000 * repetition + (int(task) if task.isdigit() else 99)

PARTIAL = CACHE / "diagnostics" / "eurosat_probe_partial.jsonl"
PARTIAL.parent.mkdir(parents=True, exist_ok=True)

results = []
if store.info(RESUME_KEY) is not None:
    # One JSON object per line, so a half-written final line can be dropped
    # without losing everything before it.
    saved = store.download(RESUME_KEY, CACHE / "cache")
    for line in saved.read_text(encoding="utf-8").splitlines():
        try:
            results.append(json.loads(line))
        except json.JSONDecodeError:
            print("Skipped one unreadable line at the end of the saved file.")
    print(f"Resuming: {len(results)} runs already finished.")
else:
    print("Starting fresh: nothing saved in R2 yet.")

PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results), encoding="utf-8")
done = {identity(r) for r in results}
new_runs = 0

def grid_at(seed, point, restore_source, repetitions):
    """Every probe task, rate, arm and repetition at one probe point."""
    global new_runs
    todo = [(task, rate, arm, code_only, train_code, repetition)
            for task in PROBE_TASKS
            for rate in RATES
            for arm, code_only, train_code in ARMS
            for repetition in range(repetitions)
            if (seed, point, task, rate, arm, repetition) not in done]
    if not todo:
        print(f"seed {seed}, t = {point}: already finished, skipping")
        return

    target, hypernet, trainer = build(seed, restore_source)
    reference = snapshot(hypernet, trainer)
    bar = tqdm(todo, desc=f"seed {seed}  t={point}", unit="run")
    for task, rate, arm, code_only, train_code, repetition in bar:
        bar.set_postfix_str(f"{task} lr={rate:g} {arm} rep {repetition + 1}")
        started = time.perf_counter()
        run = probe_once(
            hypernet, target, trainer, task=task,
            train_set=probe_sets[task]["train"], test_set=probe_sets[task]["test"],
            validation_set=probe_sets[task]["val"],
            steps=STEPS, learning_rate=rate, batch_size=BATCH_SIZE,
            code_only=code_only, train_code=train_code,
            seed=probe_seed(task, repetition), feature_images=FEATURE_IMAGES)
        if not unchanged(hypernet, reference):
            raise RuntimeError("A probe changed the model. Stop and fix this "
                               "before trusting anything.")
        run.update(point=point, endpoint="E2", repetition=repetition,
                   seed_label=seed, minutes=(time.perf_counter() - started) / 60,
                   device=GPU_NAME, run_label=RUN_LABEL,
                   train_images=len(probe_sets[task]["train"]))
        results.append(run)
        done.add(identity(run))
        # Append rather than rewrite, so saving stays cheap as the file grows.
        with PARTIAL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(run) + "\n")
        new_runs += 1
        if new_runs % SAVE_EVERY == 0:
            store.upload(PARTIAL, RESUME_KEY, overwrite=True)
    bar.close()

    del trainer, hypernet, target
    import gc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    store.upload(PARTIAL, RESUME_KEY, overwrite=True)

began = time.perf_counter()
for seed in SEEDS:
    grid_at(seed, 0, restore_source=False, repetitions=REPETITIONS_AT_ZERO)
    grid_at(seed, 8, restore_source=True, repetitions=REPETITIONS_LATER)
store.upload(PARTIAL, RESUME_KEY, overwrite=True)

print(f"\n{len(results)} runs in total, {new_runs} in this session "
      f"({(time.perf_counter() - began) / 60:.1f} minutes).")
print("The model was verified unchanged after every one.")
missing = [(s, p) for s in SEEDS for p in (0, 8)
           if not any(r["seed_label"] == s and r["point"] == p for r in results)]
if missing:
    print("Still to do, as (seed, probe point):", missing)
    print("Run this cell again to continue from here.")

## 6. The result

For each probe point, task and arm the grid keeps one learning rate per repetition: the one
with the highest **validation** accuracy, ties broken by the lower rate. Only then is that
run's **test** accuracy read. The rate is never chosen on the number the result is reported
from. Those test accuracies are then averaged over the repetitions.

The noise floor comes from the spread of the repetitions at t = 0, by the adapter study's
rule:

    d = 1.96 * s * sqrt(1/n_t + 1/n_0)

A change smaller than that is inside the measurement's own wobble and is reported as
unresolved, not as a result.

In [ ]:
import statistics

POINTS = [0, 8]

def select(runs):
    """The run with the best validation accuracy. Ties go to the lower rate."""
    return min(runs, key=lambda r: (-r["val_accuracy"], r["learning_rate"]))

def per_repetition(seed, point, task, arm):
    """One number per repetition: the test accuracy of the selected run."""
    values = []
    for repetition in range(max(REPETITIONS_AT_ZERO, REPETITIONS_LATER)):
        runs = [r for r in results
                if r["seed_label"] == seed and r["point"] == point
                and r["task"] == task and r["arm"] == arm
                and r["repetition"] == repetition]
        if runs:
            values.append(select(runs)["test_accuracy"])
    return values

SEEDS_DONE = [s for s in SEEDS
              if all(per_repetition(s, p, PROBE_TASKS[0], "full") for p in POINTS)]
print("Seeds with both probe points finished:", SEEDS_DONE or "none yet")

verdicts = {}
for seed in SEEDS_DONE:
    print(f"\n{'=' * 64}\nSeed {seed}")
    for arm, _, _ in ARMS:
        print(f"\n  arm: {arm}")
        print(f"  {'task':<10} {'t=0':>8} {'t=8':>8} {'change':>8} {'floor':>8}  verdict")
        for task in PROBE_TASKS:
            at = {p: per_repetition(seed, p, task, arm) for p in POINTS}
            spread = statistics.stdev(at[0]) if len(at[0]) > 1 else float("nan")
            floor = noise_floor(spread, spread, len(at[0]), len(at[8]))
            change = statistics.mean(at[8]) - statistics.mean(at[0])
            if change > floor:
                verdict = "higher than at t=0"
            elif change < -floor:
                verdict = "LOWER, plasticity loss"
            else:
                verdict = "unresolved, inside the floor"
            print(f"  {task:<10} {statistics.mean(at[0]):>8.2f} {statistics.mean(at[8]):>8.2f} "
                  f"{change:>+8.2f} {floor:>8.2f}  {verdict}")
            verdicts[(seed, arm, task)] = {"t0": statistics.mean(at[0]),
                                           "t8": statistics.mean(at[8]),
                                           "change": change, "floor": floor,
                                           "spread_at_zero": spread,
                                           "verdict": verdict}

# What the two tasks say together, which is the question this notebook exists for.
for seed in SEEDS_DONE:
    if not (REFERENCE_TASKS and INCLUDE_EUROSAT):
        continue
    unrelated = verdicts[(seed, "full", EUROSAT)]
    related = verdicts[(seed, "full", REFERENCE_TASKS[0])]
    print(f"\n{'=' * 64}\nSeed {seed}, reading the two together")
    print(f"  EuroSAT (unrelated):          {unrelated['change']:+.2f} "
          f"against a floor of {unrelated['floor']:.2f}")
    print(f"  Tiny ImageNet task {REFERENCE_TASKS[0]} (related): {related['change']:+.2f} "
          f"against a floor of {related['floor']:.2f}")
    if unrelated["change"] > unrelated["floor"]:
        print("  Reading: the gain survives an unrelated probe, so it is plasticity.")
    elif related["change"] > related["floor"]:
        print("  Reading: the gain appears only on the related probe, so it is transfer.")
    else:
        print("  Reading: neither probe moves past its floor. Nothing is settled here.")

## 7. Save

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
seed_label = "seeds" + "".join(str(s) for s in SEEDS_DONE)
EXPORT = CACHE / "diagnostics" / f"eurosat_probe_{RUN_LABEL}_{ORDER}_{seed_label}_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "35_eurosat_probe",
    "question": ("Is the plasticity gain measured in notebook 29 real plasticity, or "
                 "transfer from Tiny ImageNet classes the stream already saw?"),
    "protocol": ("Fork and discard, as in reference/adapter-plasticity-study. The model "
                 "is snapshotted, a fresh task code is trained on an unseen task, the "
                 "result is measured, and the model is restored and verified unchanged. "
                 "The learning rate is selected on a validation split and the result is "
                 "read from a separate test split, as the 22 October plan requires."),
    "limitations": [
        "EuroSAT is satellite imagery, so a low score at both probe points may be the "
        "domain gap rather than plasticity. The comparison is EuroSAT against itself "
        "across probe points, never against the Tiny ImageNet probe.",
        "EuroSAT has ten classes in total, so it is exactly one task and cannot be a "
        "stream.",
        "Two probe points only, 0 and 8 tasks.",
        "The two probe tasks are scored on different numbers of test images, 2,000 for "
        "EuroSAT and 500 for the Tiny ImageNet task, so their spreads are not directly "
        "comparable. Each task is compared against itself across the probe points.",
        "The first EuroSAT run, on 28 October, selected rates on test accuracy and used "
        "different split sizes, so its numbers do not belong in the same table as these.",
        "Runs saved under different RUN_LABEL values used different settings and must not "
        "be pooled. The label is recorded with every run for this reason.",
    ],
    "commit": HEAD,
    "device": GPU_NAME,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "settings": {"run_label": RUN_LABEL, "include_eurosat": INCLUDE_EUROSAT,
                 "study": STUDY_ID, "order": ORDER, "seeds": SEEDS,
                 "probe_tasks": PROBE_TASKS, "rates": RATES, "steps": STEPS,
                 "batch_size": BATCH_SIZE, "arms": [a[0] for a in ARMS],
                 "repetitions": [REPETITIONS_AT_ZERO, REPETITIONS_LATER],
                 "rate_selection": "highest validation accuracy, ties to the lower rate",
                 "split_seed": SPLIT_SEED,
                 "train_per_class": TRAIN_PER_CLASS, "val_per_class": VAL_PER_CLASS,
                 "test_per_class": TEST_PER_CLASS},
    "verdicts": {f"seed{seed}|{arm}|{task}": value
                 for (seed, arm, task), value in verdicts.items()},
}
(EXPORT / "summary.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

columns = ["run_label", "seed_label", "point", "task", "arm", "repetition",
           "learning_rate", "train_images", "val_accuracy", "val_loss",
           "test_accuracy", "test_loss", "final_train_loss", "effective_rank",
           "minutes", "device"]
with (EXPORT / "runs.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)

SAVE_VERIFIED = all((EXPORT / name).exists() and (EXPORT / name).stat().st_size > 0
                    for name in ("summary.json", "runs.csv"))
if SAVE_TO_R2 and SAVE_VERIFIED:
    for name in ("summary.json", "runs.csv"):
        store.upload(EXPORT / name, f"uncle/eurosat_probe/{STUDY_ID}/{EXPORT.name}/{name}",
                     overwrite=True)
SAVED_LOCATION = EXPORT
print("Saved to:", EXPORT)
print("Verified on disk:", SAVE_VERIFIED)

## 8. Release the GPU

Only once every file has been written and read back. If anything failed to save, the
runtime stays connected so the probe runs are not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or empty.")
    print("Rerun section 7 before closing this runtime, or the probe runs are lost.")
elif missing:
    print("Results so far saved to:", SAVED_LOCATION)
    print("NOT releasing the GPU. These are not finished yet:", missing)
    print("Run the grid cell again to continue from where it stopped.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Everything saved. Leaving the runtime connected as asked.")
else:
    print("Everything saved. Releasing the GPU.")
    if IN_COLAB:
        from google.colab import runtime
        runtime.unassign()